# StockSense — Phase 3: Feature Engineering & Demand Intelligence
**Project:** IntelliData 2026 StockSense Challenge  
**Phase:** 3 (Feature Engineering, Leakage Prevention & Dataset Construction)  
**Objective:** Transform the Phase 1 cleaned master dataset into a leakage-safe, feature-rich dataset supporting 7-day demand forecasting and stock-out classification.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# Set working directory and paths
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.features.build_features import build_feature_pipeline
print(f"StockSense Root: {ROOT}")


## 1. Execute Modular Feature Engineering Pipeline
The pipeline imports reusable functions from `src/features/` and constructs Group A through Group P features.

In [ ]:
master_feat = pd.read_csv(ROOT / 'data' / 'processed' / 'features' / 'feature_master.csv', parse_dates=['date'])
print(f"Engineered Master Shape: {master_feat.shape[0]:,} rows x {master_feat.shape[1]} columns")
master_feat.head(3)


## 2. Programmatic Leakage Verification
Verify that rolling windows strictly exclude day $t$'s demand.

In [ ]:
# Verify rolling window shifted integrity on sample
sample = master_feat[(master_feat['store_id'] == 'S01') & (master_feat['product_id'] == 'P101')].sort_values('date')
print("Sample verification for Store S01, Product P101:")
sample[['date', 'transaction_demand', 'lag_1', 'lag_7', 'rolling_mean_7', 'next_7_day_demand']].head(10)


## 3. Demand Intelligence: Momentum, Volatility & Regimes

In [ ]:
print("Demand Regime Breakdown:")
print(master_feat['demand_regime'].value_counts())

fig, ax = plt.subplots(figsize=(8, 4))
sns.countplot(data=master_feat, x='demand_regime', palette='viridis', ax=ax)
ax.set_title('Distribution of Store x Product Demand Regimes')
plt.tight_layout()
plt.show()


## 4. Inventory Intelligence: Days to Stockout & Lead-Time Gap

In [ ]:
inv_valid = master_feat[master_feat['days_to_stockout'].notna()]
print("Inventory Intelligence Summary:")
print(inv_valid[['days_to_stockout', 'lead_time_gap', 'shelf_life_exposure_ratio']].describe())

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(inv_valid['lead_time_gap'].clip(lower=-20, upper=50), bins=35, kde=True, ax=ax, color='crimson')
ax.axvline(0, color='black', linestyle='--', label='Replenishment Parity')
ax.set_title('Lead Time Gap Distribution (Days to Stockout - Lead Days)')
ax.legend()
plt.tight_layout()
plt.show()


## 5. Target Variable Validation
Validation of `next_7_day_demand` (regression target) and `stockout_flag` (classification target).

In [ ]:
print("Target Availability:")
print(f"next_7_day_demand valid rows: {master_feat['next_7_day_demand'].notna().sum():,} / {len(master_feat):,}")
print(f"stockout_flag valid rows: {master_feat['stockout_flag'].notna().sum():,} / {len(master_feat):,}")
print(f"stockout_flag positive stockouts: {(master_feat['stockout_flag'] == 1).sum()}")

# Check chronological train/val/test splits
demand_dates = master_feat[master_feat['next_7_day_demand'].notna()]['date']
print(f"Valid demand target dates: {demand_dates.min().strftime('%Y-%m-%d')} to {demand_dates.max().strftime('%Y-%m-%d')}")
